# 04 -- Krzywa uporzadkowana: przygotowanie i sredni rok

Ten notebook obejmuje:

1. **Filtrowanie lat** -- usuniecie lat powodziowych i niekompletnych
2. **Krzywa uporzadkowana (FDC)** -- na danych przefiltrowanych
3. **Sredni rok uporzadkowany** -- sortowanie kazdego roku, usrednienie po pozycjach

---

## Konfiguracja

### Modul `src/hydrology.py` -- nowe funkcje

**Prompt do LLM tworzacy te funkcje:**
> *"Rozszerz modul src/hydrology.py o:
> 1. Obliczanie kompletnosci danych per rok (ile dni ma dane).
> 2. Identyfikacje lat powodziowych (roczne max >> srednia rocznych max * prog).
> 3. Filtrowanie lat -- usuwanie wybranych lub niekompletnych lat z DataFrame.
> 4. Sredni rok uporzadkowany -- dla kazdego roku posortuj przeplywy malejaco (365 wartosci),
>    potem dla kazdej pozycji (1..365) oblicz srednia, min, max po wszystkich latach.
>    To daje wygladzona typowa krzywa uporzadkowana."*

**Nowe funkcje w `src/hydrology.py`:**
- `year_completeness(df, station_id)` -- kompletnosc per rok
- `identify_flood_years(df, station_id, threshold_factor)` -- lista lat powodziowych
- `filter_years(df, station_id, years_to_remove)` -- usun wybrane lata
- `filter_incomplete_years(df, station_id, min_completeness)` -- usun niekompletne lata
- `average_sorted_year(df, station_id)` -- sredni rok uporzadkowany

**Prompt do LLM:**
> *"Napisz kod konfiguracji notebooka: zaladuj modul src/imgw_data (load_processed) i src/hydrology
> (year_completeness, identify_flood_years, filter_years, filter_incomplete_years,
> flow_duration_curve, characteristic_flows, average_sorted_year)."*

**Uzyte moduly/funkcje:** `src.imgw_data`: load_processed; `src.hydrology`: year_completeness, identify_flood_years, filter_years, filter_incomplete_years, flow_duration_curve, characteristic_flows, average_sorted_year

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.imgw_data import load_processed
from src.hydrology import (
    year_completeness,
    identify_flood_years,
    filter_years,
    filter_incomplete_years,
    flow_duration_curve,
    characteristic_flows,
    average_sorted_year,
)

pd.set_option('display.max_columns', 15)
print('Moduly zaladowane.')

**Prompt do LLM:**
> *"Napisz kod ktory uzyje load_processed() z modulu src/imgw_data do wczytania danych.
> Wybierz stacje glowna."*

**Uzyte moduly/funkcje:** `src.imgw_data.load_processed()`

In [ ]:
# Wczytaj dane
df = load_processed('../data/processed/daily_hydro_clean.parquet')
PRIMARY_STATION = df['station_id'].unique()[0]
name = df.loc[df['station_id'] == PRIMARY_STATION, 'station_name'].iloc[0]
river = df.loc[df['station_id'] == PRIMARY_STATION, 'river_name'].iloc[0]
print(f'Stacja: {name} ({PRIMARY_STATION}) na rzece {river}')
print(f'Okres: {df["date"].min()} -- {df["date"].max()}')
print(f'Liczba wierszy: {len(df)}')

---
## Krok 1: Analiza kompletnosci lat

Przed budowa krzywej uporzadkowanej musimy sprawdzic, ktore lata maja kompletne dane.
Lata niekompletne (duzo brakow) moga znieksztalcic wyniki.

**Prompt do LLM:**
> *"Napisz kod ktory uzyje funkcji year_completeness() z modulu src/hydrology
> do obliczenia kompletnosci danych per rok. Wyswietl tabelke i narysuj wykres slupkowy
> procentu kompletnosci."*

**Uzyte moduly/funkcje:** `src.hydrology.year_completeness()`

In [ ]:
# Kompletnosc per rok
comp = year_completeness(df, PRIMARY_STATION)
print(f'Lata z danymi: {len(comp)}')
print(comp.to_string())

In [ ]:
# Wykres kompletnosci
fig = go.Figure()
fig.add_trace(go.Bar(
    x=comp.index, y=comp['pct_valid'],
    marker_color=['green' if p >= 95 else 'orange' if p >= 80 else 'red' for p in comp['pct_valid']],
))
fig.add_hline(y=95, line_dash='dash', line_color='green', annotation_text='95%')
fig.update_layout(
    title=f'Kompletnosc danych per rok -- {name}',
    xaxis_title='Rok', yaxis_title='Kompletnosc [%]',
    yaxis_range=[0, 105], height=400,
)
fig.show()

---
## Krok 2: Identyfikacja lat powodziowych

Lata z ekstremalnymi powodziami moga zawyzyc srednia krzywa uporzadkowana.
Identyfikujemy je automatycznie (roczne max > prog * SWQ) i dajemy mozliwosc recznego usuniecia.

**Prompt do LLM:**
> *"Napisz kod ktory uzyje identify_flood_years() z modulu src/hydrology do znalezienia
> lat powodziowych. Pokaz roczne maksima na wykresie z linia progu."*

**Uzyte moduly/funkcje:** `src.hydrology.identify_flood_years()`

In [ ]:
# Identyfikacja lat powodziowych (prog: 3x SWQ)
flood_years = identify_flood_years(df, PRIMARY_STATION, threshold_factor=3.0)
print(f'Lata powodziowe (maks > 3x SWQ): {flood_years}')

# Wykres rocznych maksimow
sdf = df[df['station_id'] == PRIMARY_STATION].dropna(subset=['discharge_m3s']).copy()
sdf['year'] = sdf['date'].dt.year
annual_max = sdf.groupby('year')['discharge_m3s'].max()
swq = annual_max.mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=annual_max.index, y=annual_max.values,
    marker_color=['red' if y in flood_years else 'royalblue' for y in annual_max.index],
    name='Roczne max Q',
))
fig.add_hline(y=swq, line_dash='dash', line_color='black', annotation_text=f'SWQ = {swq:.1f} m3/s')
fig.add_hline(y=3*swq, line_dash='dash', line_color='red', annotation_text=f'3x SWQ = {3*swq:.1f} m3/s')
fig.update_layout(
    title=f'Roczne maksima przeplywu -- {name}',
    xaxis_title='Rok', yaxis_title='Q max [m3/s]', height=400,
)
fig.show()

---
## Krok 3: Filtrowanie danych

Usuwamy lata niekompletne i (opcjonalnie) powodziowe.
Mozna tez recznie dodac lata do usuniecia.

**Prompt do LLM:**
> *"Napisz kod ktory: (1) usunie lata niekompletne (<95%) uzywajac filter_incomplete_years(),
> (2) da mozliwosc recznego dodania lat do usuniecia (np. powodziowych),
> (3) pokaze ile lat zostalo po filtrowaniu."*

**Uzyte moduly/funkcje:** `src.hydrology.filter_incomplete_years()`, `src.hydrology.filter_years()`

In [ ]:
# Krok 1: Usun lata niekompletne (<95% danych)
df_filtered = filter_incomplete_years(df, PRIMARY_STATION, min_completeness=0.95)

# Krok 2: Opcjonalnie usun lata powodziowe (odkomentuj jezeli potrzeba)
# YEARS_TO_REMOVE = flood_years  # lub recznie: [1997, 2010]
# df_filtered = filter_years(df_filtered, PRIMARY_STATION, YEARS_TO_REMOVE)

# Podsumowanie
n_before = df[df['station_id'] == PRIMARY_STATION]['date'].dt.year.nunique()
n_after = df_filtered[df_filtered['station_id'] == PRIMARY_STATION]['date'].dt.year.nunique()
print(f'Lata przed filtrowaniem: {n_before}')
print(f'Lata po filtrowaniu: {n_after}')
print(f'Usunieto: {n_before - n_after} lat')

---
## Krok 4: Krzywa uporzadkowana na danych przefiltrowanych

Porownanie FDC przed i po filtrowaniu.

**Prompt do LLM:**
> *"Napisz kod ktory obliczy krzywa uporzadkowana (FDC) z flow_duration_curve()
> na danych oryginalnych i przefiltrowanych, i narysuje obie na jednym wykresie
> z osia Y logarytmiczna."*

**Uzyte moduly/funkcje:** `src.hydrology.flow_duration_curve()`

In [ ]:
# FDC przed i po filtrowaniu
fdc_orig = flow_duration_curve(df, PRIMARY_STATION)
fdc_filt = flow_duration_curve(df_filtered, PRIMARY_STATION)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fdc_orig['exceedance_pct'], y=fdc_orig['discharge_m3s'],
    mode='lines', name='Oryginalne', line=dict(color='gray', width=1),
))
fig.add_trace(go.Scatter(
    x=fdc_filt['exceedance_pct'], y=fdc_filt['discharge_m3s'],
    mode='lines', name='Przefiltrowane', line=dict(color='royalblue', width=2),
))
fig.update_layout(
    title=f'Krzywa uporzadkowana -- {name}',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]', yaxis_type='log',
    height=500, hovermode='x unified',
)
fig.show()

---
## Krok 5: Sredni rok uporzadkowany

Dla kazdego roku:
1. Sortujemy dobowe przeplywy od najwyzszego do najnizszego (365 wartosci)
2. Przypisujemy pozycje (rang) 1..365

Nastepnie dla kazdej pozycji obliczamy srednia, min i max po wszystkich latach.
Daje to **wygladzona, typowa krzywa uporzadkowana** -- przydatna do projektowania.

**Prompt do LLM:**
> *"Napisz kod ktory uzyje average_sorted_year() z modulu src/hydrology do obliczenia
> sredniego roku uporzadkowanego na danych przefiltrowanych. Narysuj wykres: srednia
> jako linia, zakres min-max jako wypelnienie, os X to procent przekroczenia."*

**Uzyte moduly/funkcje:** `src.hydrology.average_sorted_year()`

In [ ]:
# Sredni rok uporzadkowany
avg_year = average_sorted_year(df_filtered, PRIMARY_STATION)
print(f'Obliczono z {avg_year["n_years"].iloc[0]} lat')
avg_year.head(10)

In [ ]:
# Wykres sredniego roku uporzadkowanego
fig = go.Figure()

# Zakres min-max
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['max'],
    mode='lines', line=dict(width=0), showlegend=False,
))
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['min'],
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(65,105,225,0.15)',
    name='Zakres min-max',
))

# Srednia
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['mean'],
    mode='lines', line=dict(color='royalblue', width=2.5),
    name='Srednia',
))

# Przeplywy charakterystyczne
chars = characteristic_flows(df_filtered, PRIMARY_STATION)
for q_name, color in [('Q50', 'green'), ('Q90', 'orange'), ('Q95', 'red')]:
    q_val = chars[q_name]
    fig.add_hline(y=q_val, line_dash='dash', line_color=color,
                  annotation_text=f'{q_name} = {q_val:.2f} m3/s')

fig.update_layout(
    title=f'Sredni rok uporzadkowany -- {name} ({avg_year["n_years"].iloc[0]} lat)',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]',
    height=500, hovermode='x unified',
)
fig.show()

In [ ]:
# Ten sam wykres w skali logarytmicznej
fig.update_layout(yaxis_type='log', title=f'Sredni rok uporzadkowany (log) -- {name}')
fig.show()

---
## Krok 6: Porownanie FDC i sredniego roku

FDC ze wszystkich danych vs sredni rok uporzadkowany -- powinny byc zblizone,
ale sredni rok jest bardziej wygladzony.

**Prompt do LLM:**
> *"Narysuj na jednym wykresie: FDC z przefiltrowanych danych i sredni rok uporzadkowany.
> Porownaj ksztalty."*

**Uzyte moduly/funkcje:** `src.hydrology.flow_duration_curve()`, `src.hydrology.average_sorted_year()`

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fdc_filt['exceedance_pct'], y=fdc_filt['discharge_m3s'],
    mode='lines', name='FDC (wszystkie dane)', line=dict(color='gray', width=1),
))
fig.add_trace(go.Scatter(
    x=avg_year['exceedance_pct'], y=avg_year['mean'],
    mode='lines', name='Sredni rok uporzadkowany', line=dict(color='royalblue', width=2.5),
))
fig.update_layout(
    title=f'Porownanie: FDC vs sredni rok uporzadkowany -- {name}',
    xaxis_title='Prawdopodobienstwo przekroczenia [%]',
    yaxis_title='Q [m3/s]', yaxis_type='log',
    height=500, hovermode='x unified',
)
fig.show()

---
## Podsumowanie

W tym notebooku:
1. Sprawdzilismy kompletnosc danych per rok
2. Zidentyfikowalismy lata powodziowe
3. Przefiltrowalismy dane (usunelismy niekompletne/powodziowe lata)
4. Porownalismy krzywa uporzadkowana przed i po filtrowaniu
5. Obliczylismy sredni rok uporzadkowany -- wygladzona typowa krzywa

**Sredni rok uporzadkowany** jest podstawa do dalszych obliczen:
- Szacowanie produkcji energii (pole pod krzywa)
- Dobor turbiny (zakres przeplywow eksploatacyjnych)

**Dalej:** obliczanie potencjalu energetycznego